# Results Analysis — Article Generation Pipeline

Methodical analysis of pipeline runs (per §9 of the guidelines): token/cost
trends, validator pass-rate over time, and the benchmark figure's data spec.

Data sources (read-only):
- `logs/app.log` — per-run TOKEN USAGE & COST blocks and validation results
- `outputs/assets/graph_spec.json` — the benchmark figure's three-architecture spec

Run top-to-bottom. No API calls are made; this only parses existing artifacts.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt

# Resolve the project root whether the notebook runs from repo root or notebooks/.
ROOT = Path.cwd()
while not (ROOT / "logs" / "app.log").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("project root:", ROOT)

## 1. Token usage & cost per run

Parsed from the `TOKEN USAGE & COST` blocks the pipeline logs after every
`crew.kickoff`.

In [ ]:
log_text = (ROOT / "logs" / "app.log").read_text(encoding="utf-8", errors="replace")

runs = [
    {"prompt": int(m.group(1)), "output": int(m.group(2)),
     "total": int(m.group(3)), "cost": float(m.group(4))}
    for m in re.finditer(
        r"prompt tokens\s*:\s*(\d+).*?output tokens\s*:\s*(\d+)"
        r".*?total tokens\s*:\s*(\d+).*?estimated cost\s*:\s*\$([\d.]+)",
        log_text, re.DOTALL)
]
print(f"parsed {len(runs)} runs")
if runs:
    last = runs[-1]
    print(f"latest run: {last['total']:,} tokens, ${last['cost']:.4f}")

In [ ]:
idx = list(range(1, len(runs) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(idx, [r["prompt"] for r in runs], marker="o", label="prompt")
ax1.plot(idx, [r["output"] for r in runs], marker="s", label="output")
ax1.plot(idx, [r["total"] for r in runs], marker="^", label="total")
ax1.set_title("Token usage per run")
ax1.set_xlabel("run #"); ax1.set_ylabel("tokens"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(idx, [r["cost"] for r in runs], marker="o", color="crimson")
ax2.set_title("Estimated cost per run (USD)")
ax2.set_xlabel("run #"); ax2.set_ylabel("$"); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 2. Validator pass-rate over time

Parsed from the `Result: N/13 passed` lines. Tracks how reliably a run satisfies
the 13-item assignment checklist.

In [ ]:
results = [(int(p), int(t)) for p, t in re.findall(r"Result:\s*(\d+)/(\d+)\s*passed", log_text)]
passed = [p for p, _ in results]
total_checks = results[0][1] if results else 13

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, len(passed) + 1), passed, marker="o")
ax.axhline(total_checks, ls="--", color="green", label=f"all {total_checks} pass")
ax.set_title("Validator checks passed per run")
ax.set_xlabel("run #"); ax.set_ylabel("checks passed")
ax.set_ylim(0, total_checks + 1); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

if passed:
    perfect = sum(1 for p in passed if p == total_checks)
    print(f"{perfect}/{len(passed)} runs were perfect ({total_checks}/{total_checks})")

## 3. Benchmark figure data spec

The three-architecture performance spec that drives the generated benchmark
figure, with provenance (`data_basis` = measured vs. estimated).

In [ ]:
spec_path = ROOT / "outputs" / "assets" / "graph_spec.json"
if spec_path.exists():
    spec = json.loads(spec_path.read_text(encoding="utf-8"))
    cols = ("name", "median_queue", "p95_queue", "base_fct_ms", "fct_slope", "data_basis")
    print(f"{'role':<8}" + "".join(f"{c:<14}" for c in cols))
    for role in ("main", "arch_a", "arch_b"):
        s = spec[role]
        print(f"{role:<8}" + "".join(f"{str(s.get(c, '')):<14}" for c in cols))
else:
    print("No graph_spec.json yet — run the pipeline first.")